# 3Di Factory: query AA → homolog AA → ProstT5 3Di

This notebook takes each benchmark query protein, obtains an MMseqs2/ColabFold A3M family alignment, removes the query itself (including exact-sequence duplicates), translates selected homolog AA sequences to 3Di with the full ProstT5 encoder-decoder model, projects each homolog's 3Di states onto query coordinates, and saves each query immediately after it is converted.

Outputs are saved under `prostT5_benchmarks/folding_MSA/<query_id>/`, next to the cached source A3M, so the job can be resumed or split across machines:

- raw homolog AA FASTA
- raw homolog 3Di FASTA (one state per homolog residue)
- query-coordinate projected homolog 3Di FASTA (`-` where a homolog has no residue)
- metadata CSV and the cached source A3M


## 1. Imports and configuration

All queries are loaded directly from the repository's 100-protein benchmark FASTA. There is no single-query name or pasted-sequence input.

In [1]:
from pathlib import Path
import io
import os
import shutil
import tarfile
import time
import zipfile

import numpy as np
import pandas as pd
import requests
import torch
from transformers import AutoModelForSeq2SeqLM, T5Tokenizer
from tqdm.auto import tqdm

PROSTT5_NAME = "Rostlab/ProstT5_fp16"
BENCHMARK_FASTA = Path("prostT5_benchmarks/benchmark_data/test_set_AA.fasta")
BENCHMARK_FASTA_URL = (
    "https://raw.githubusercontent.com/Viet2503/Speculative-Decoding-ProstT5/"
    "HMM-folding/prostT5/prostT5_benchmarks/benchmark_data/test_set_AA.fasta"
)

MAX_HOMOLOGS = int(os.environ.get("THREEDI_FACTORY_MAX_HOMOLOGS", "64"))
MAX_QUERIES_TO_PROCESS = int(os.environ.get("THREEDI_FACTORY_MAX_QUERIES", "0")) or None
MIN_HOMOLOG_AA_LENGTH = 40
MIN_QUERY_COVERAGE = 0.50
MSA_MODE = "env"
MSA_SUBDIR = Path("prostT5_benchmarks/folding_MSA")
LOCAL_PROST_DIR = Path("/Users/chencheng-lin/Desktop/Speculative-Decoding-ProstT5/prostT5")
LOCAL_MSA_DIR = Path("/Users/chencheng-lin/Desktop/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA")
COLAB_MSA_DIR = Path(os.environ.get(
    "THREEDI_FACTORY_COLAB_MSA_DIR",
    "/content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA",
))
OUTPUT_MODE = os.environ.get("THREEDI_FACTORY_OUTPUT_MODE", "auto").lower()
REQUIRE_LOCAL_MAC_OUTPUT = os.environ.get("THREEDI_FACTORY_REQUIRE_LOCAL", "0") == "1"

AA_LETTERS = set("ACDEFGHIKLMNPQRSTVWY")
THREEDI_ALPHABET = "acdefghiklmnpqrstvwy"

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Device: {device}")


Device: cuda


## 2. Resolve repository paths and load all 100 benchmark queries

The benchmark FASTA is resolved directly under `PROST_DIR`; every FASTA record becomes one query.

In [2]:
def parse_fasta(path: Path) -> dict[str, str]:
    records = {}
    name = None
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line:
            continue
        if line.startswith(">"):
            name = line[1:].split()[0]
            records[name] = ""
        elif name is not None:
            records[name] += line
    return records


def normalize_aa(seq: str) -> str:
    seq = "".join(seq.split()).upper()
    bad = sorted(set(seq) - AA_LETTERS)
    if bad:
        raise ValueError(f"Unsupported AA symbols: {bad}")
    return seq


def has_benchmark_data(prost_dir: Path) -> bool:
    data_dir = prost_dir / "prostT5_benchmarks" / "benchmark_data"
    return (data_dir / "test_set_AA.fasta").exists()


def candidate_prost_dirs(start: Path) -> list[Path]:
    candidates = [LOCAL_PROST_DIR]
    for env_name in ("PROSTT5_DIR", "PROST_DIR"):
        if os.environ.get(env_name):
            candidates.append(Path(os.environ[env_name]).expanduser())
    for base in (start, *start.parents):
        candidates.extend([base, base / "prostT5"])
    candidates.extend([
        Path("/content/drive/MyDrive"),
        Path("/content/drive/MyDrive/prostT5"),
        Path("/content/Speculative-Decoding-ProstT5/prostT5"),
        Path("/content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5"),
    ])
    unique = []
    seen = set()
    for candidate in candidates:
        resolved = candidate.expanduser().resolve()
        if resolved not in seen:
            seen.add(resolved)
            unique.append(resolved)
    return unique


def materialize_benchmark_fasta() -> Path:
    """Download the repository's 100-protein AA FASTA when Colab has no checkout/Drive data."""
    fallback_dir = Path("/content/prostT5") if Path("/content").exists() else Path.cwd() / ".prostT5_data"
    fasta_path = fallback_dir / BENCHMARK_FASTA
    fasta_path.parent.mkdir(parents=True, exist_ok=True)
    response = requests.get(BENCHMARK_FASTA_URL, timeout=60)
    response.raise_for_status()
    text = response.text
    if text.count(">") != 100:
        raise RuntimeError(
            f"Downloaded benchmark should contain 100 FASTA records, found {text.count('>')}"
        )
    fasta_path.write_text(text, encoding="utf-8")
    print(f"Downloaded benchmark FASTA to {fasta_path}")
    return fallback_dir.resolve()


NOTEBOOK_DIR = Path.cwd().resolve()
PROST_DIR = next(
    (candidate for candidate in candidate_prost_dirs(NOTEBOOK_DIR) if has_benchmark_data(candidate)),
    None,
)
if PROST_DIR is None:
    PROST_DIR = materialize_benchmark_fasta()
benchmark_fasta_path = PROST_DIR / BENCHMARK_FASTA
benchmark_queries = {
    query_id: normalize_aa(sequence)
    for query_id, sequence in parse_fasta(benchmark_fasta_path).items()
}
if len(benchmark_queries) != 100:
    raise ValueError(f"Expected 100 benchmark proteins, found {len(benchmark_queries)}")

def running_in_colab() -> bool:
    return Path("/content").exists() or str(NOTEBOOK_DIR).startswith("/content")


def ensure_colab_drive_mounted() -> None:
    if not running_in_colab() or str(COLAB_MSA_DIR).startswith("/content") is False:
        return
    drive_root = Path("/content/drive/MyDrive")
    if drive_root.exists():
        return
    try:
        from google.colab import drive
    except Exception as exc:
        raise RuntimeError("Google Drive output requested, but google.colab.drive is unavailable") from exc
    drive.mount("/content/drive")


def resolve_msa_dir() -> Path:
    """Choose the real writable output folder for this runtime.

    Colab cannot write directly to `/Users/chencheng-lin/...` on the Mac. In Colab,
    save to Google Drive and then sync/download that folder to the local Mac path.
    """
    local_prost = LOCAL_PROST_DIR.expanduser().resolve()
    active_prost = PROST_DIR.expanduser().resolve()
    if REQUIRE_LOCAL_MAC_OUTPUT and active_prost != local_prost:
        raise RuntimeError(
            "Direct local-Mac output was required, but this kernel is not using the local checkout. "
            "Colab cannot write into `/Users/chencheng-lin/...` on your Mac."
        )
    if OUTPUT_MODE in {"colab", "colab_drive", "drive"} or (OUTPUT_MODE == "auto" and running_in_colab()):
        ensure_colab_drive_mounted()
        print("Colab runtime detected: saving checkpoints to Google Drive.")
        print(f"After the run, download/sync this folder to the local Mac folder: {LOCAL_MSA_DIR}")
        return COLAB_MSA_DIR
    return LOCAL_MSA_DIR


MSA_DIR = resolve_msa_dir()
MSA_DIR.mkdir(parents=True, exist_ok=True)
HOMOLOG_MANIFEST_PATH = MSA_DIR / "homologs_fetched_by_api.csv"
OUTPUT_BASE = MSA_DIR
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

print(f"ProstT5 dir: {PROST_DIR}")
print(f"Benchmark FASTA: {benchmark_fasta_path}")
print(f"Benchmark queries: {len(benchmark_queries)}")
print(f"MSA/output cache: {MSA_DIR}")
print(f"AA → 3Di translator: full {PROSTT5_NAME} encoder-decoder")
print(f"Output base: {OUTPUT_BASE}")


Mounted at /content/drive
Colab runtime detected: saving checkpoints to Google Drive.
After the run, download/sync this folder to the local Mac folder: /Users/chencheng-lin/Desktop/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA
ProstT5 dir: /content/prostT5
Benchmark FASTA: /content/prostT5/prostT5_benchmarks/benchmark_data/test_set_AA.fasta
Benchmark queries: 100
MSA/output cache: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA
AA → 3Di translator: full Rostlab/ProstT5_fp16 encoder-decoder
Output base: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA


## 3. Obtain and parse the homolog A3M

The ColabFold MMseqs2 result is already aligned in A3M format. Lowercase letters are insertions relative to the query; removing them gives the shared query-column alignment. Therefore, do not independently realign each homolog before projection.

In [3]:
COLABFOLD_HOST = "https://api.colabfold.com"
COLABFOLD_HEADERS = {"User-Agent": "prostt5-3di-factory/1.0 (educational use)"}
RETRY_STATUS = {429, 500, 502, 503, 504}


def cf_request(method: str, url: str, max_retries: int = 8, **kwargs) -> requests.Response:
    kwargs.setdefault("headers", COLABFOLD_HEADERS)
    response = None
    for attempt in range(max_retries):
        response = requests.request(method, url, **kwargs)
        if response.status_code not in RETRY_STATUS:
            response.raise_for_status()
            return response
        wait = min(10.0 * (2 ** attempt), 120.0)
        if response.headers.get("Retry-After"):
            try:
                wait = float(response.headers["Retry-After"])
            except ValueError:
                pass
        print(f"ColabFold status {response.status_code}; retrying in {wait:.0f}s")
        time.sleep(wait)
    response.raise_for_status()
    return response


def fetch_msa_colabfold(aa_seq: str, cache_path: Path, mode: str = "env",
                       poll_interval: float = 5.0, max_wait_s: float = 900.0,
                       force_refresh: bool = False) -> str:
    if cache_path.exists() and not force_refresh:
        print(f"Using cached A3M: {cache_path}")
        return cache_path.read_text(encoding="utf-8")

    submitted = cf_request(
        "POST", f"{COLABFOLD_HOST}/ticket/msa",
        data={"q": f">query\n{aa_seq}\n", "mode": mode}, timeout=30,
    ).json()
    job_id = submitted.get("id")
    status = submitted.get("status", "")
    if not job_id:
        raise RuntimeError(f"ColabFold submission failed: {submitted}")

    waited = 0.0
    while status != "COMPLETE":
        if status in {"ERROR", "MAINTENANCE"}:
            raise RuntimeError(f"ColabFold job {job_id} ended with status {status}")
        if waited >= max_wait_s:
            raise TimeoutError(f"ColabFold job {job_id} exceeded {max_wait_s}s")
        time.sleep(poll_interval)
        waited += poll_interval
        status = cf_request("GET", f"{COLABFOLD_HOST}/ticket/{job_id}", timeout=15).json().get("status", "")

    data = cf_request("GET", f"{COLABFOLD_HOST}/result/download/{job_id}", timeout=120).content
    parts = []
    if data[:2] == b"PK":
        with zipfile.ZipFile(io.BytesIO(data)) as archive:
            parts = [archive.read(name).decode("utf-8", "ignore") for name in archive.namelist() if name.endswith(".a3m")]
    elif data[:2] == b"\x1f\x8b":
        with tarfile.open(fileobj=io.BytesIO(data)) as archive:
            for member in archive.getmembers():
                if member.name.endswith(".a3m"):
                    handle = archive.extractfile(member)
                    if handle is not None:
                        parts.append(handle.read().decode("utf-8", "ignore"))
    else:
        text = data.decode("utf-8", "ignore")
        if text.lstrip().startswith(">"):
            parts = [text]
    a3m = "\n".join(part.strip() for part in parts if part.strip())
    if not a3m:
        raise RuntimeError("ColabFold result did not contain an A3M alignment")
    cache_path.write_text(a3m + "\n", encoding="utf-8")
    return a3m


def parse_a3m_records(a3m: str) -> list[tuple[str, str]]:
    records = []
    name = None
    chunks = []
    for raw in a3m.splitlines():
        line = raw.replace("\x00", "").strip()
        if not line:
            continue
        if line.startswith(">"):
            if name is not None:
                records.append((name, "".join(chunks)))
            name = line[1:].split()[0] or f"seq{len(records)}"
            chunks = []
        else:
            chunks.append(line)
    if name is not None:
        records.append((name, "".join(chunks)))
    return records


def strip_a3m_insertions(seq: str) -> str:
    return "".join(c.upper() for c in seq if not c.islower() and c not in ".*")


def ungap_aa(seq: str) -> str:
    return "".join(c for c in seq.upper() if c in AA_LETTERS)


def alignment_stats(query_aln: str, homolog_aln: str) -> tuple[float, float]:
    paired = [(q, h) for q, h in zip(query_aln, homolog_aln) if q != "-" and h != "-"]
    coverage = len(paired) / max(1, sum(q != "-" for q in query_aln))
    identity = sum(q == h for q, h in paired) / max(1, len(paired))
    return coverage, identity


def select_ranked_homologs(a3m_text: str, query_aa: str,
                            max_homologs: int | None = None) -> tuple[str, list[dict]]:
    records = parse_a3m_records(a3m_text)
    if not records:
        raise ValueError("A3M has no records")
    query_alignment = strip_a3m_insertions(records[0][1])
    if ungap_aa(query_alignment) != query_aa:
        raise ValueError("The first A3M row does not match the benchmark query")

    homologs = []
    seen_sequences = {query_aa}
    for api_rank, (name, raw_alignment) in enumerate(records[1:], start=1):
        homolog_alignment = strip_a3m_insertions(raw_alignment)
        homolog_aa = ungap_aa(homolog_alignment)
        if not homolog_aa or homolog_aa in seen_sequences:
            continue
        if len(homolog_aa) < MIN_HOMOLOG_AA_LENGTH:
            continue
        coverage, identity = alignment_stats(query_alignment, homolog_alignment)
        if coverage < MIN_QUERY_COVERAGE:
            continue
        seen_sequences.add(homolog_aa)
        homologs.append({
            "api_rank": api_rank,
            "selected_rank": len(homologs) + 1,
            "name": name,
            "aa": homolog_aa,
            "aa_alignment": homolog_alignment,
            "query_coverage": coverage,
            "query_identity": identity,
        })
        if max_homologs is not None and len(homologs) >= max_homologs:
            break
    return query_alignment, homologs


def count_fasta_records(path: Path) -> int:
    if not path.exists():
        return 0
    return sum(1 for line in path.read_text(encoding="utf-8").splitlines() if line.startswith(">"))


def converted_output_count(query_id: str) -> int:
    projected_path = OUTPUT_BASE / query_id / "homologs_projected_to_query_3di.fasta"
    return count_fasta_records(projected_path)


pending_queries = []
skipped_queries = []
for query_id, query_aa in benchmark_queries.items():
    a3m_path = MSA_DIR / f"{query_id}.a3m"
    converted_count = converted_output_count(query_id)
    cached_count = 0
    if a3m_path.exists():
        try:
            _, cached_rows = select_ranked_homologs(a3m_path.read_text(encoding="utf-8"), query_aa)
            cached_count = len(cached_rows)
        except Exception as exc:
            print(f"{query_id}: invalid cached A3M ({exc}); fetching again")
    if converted_count >= MAX_HOMOLOGS:
        skipped_queries.append({
            "query_id": query_id,
            "usable_homologs": cached_count,
            "converted_homologs": converted_count,
        })
        print(f"SKIP {query_id}: {converted_count} converted homolog 3Di rows already saved")
    else:
        pending_queries.append({
            "query_id": query_id,
            "query_aa": query_aa,
            "cached_homologs": cached_count,
            "converted_homologs": converted_count,
        })
        if MAX_QUERIES_TO_PROCESS is not None and len(pending_queries) >= MAX_QUERIES_TO_PROCESS:
            break

print(f"Complete/skipped: {len(skipped_queries)}")
print(f"Pending: {len(pending_queries)}")


SKIP P01308: 64 converted homolog 3Di rows already saved
SKIP P02798: 64 converted homolog 3Di rows already saved
SKIP P62944: 64 converted homolog 3Di rows already saved
SKIP P01542: 64 converted homolog 3Di rows already saved
SKIP P43408: 64 converted homolog 3Di rows already saved
SKIP P0AFL6: 64 converted homolog 3Di rows already saved
SKIP P0ACJ8: 64 converted homolog 3Di rows already saved
SKIP P0A988: 64 converted homolog 3Di rows already saved
SKIP P0A7Y4: 64 converted homolog 3Di rows already saved
SKIP Q57733: 64 converted homolog 3Di rows already saved
SKIP P22579: 64 converted homolog 3Di rows already saved
SKIP P15378: 64 converted homolog 3Di rows already saved
SKIP P34690: 64 converted homolog 3Di rows already saved
SKIP P0A7U3: 64 converted homolog 3Di rows already saved
SKIP P63165: 64 converted homolog 3Di rows already saved
SKIP P0A7N4: 64 converted homolog 3Di rows already saved
SKIP P24311: 64 converted homolog 3Di rows already saved
SKIP P56252: 64 converted homol

## 4. Load ProstT5 for AA → 3Di generation

This factory deliberately uses the full `Rostlab/ProstT5_fp16` encoder-decoder with the `<AA2fold>` prompt. It does **not** use `model_aa_3di.pt` or a CNN head.

In [4]:
def format_aa(seq: str) -> str:
    return "<AA2fold> " + " ".join(seq.upper())


try:
    tokenizer = T5Tokenizer.from_pretrained(PROSTT5_NAME, do_lower_case=False, legacy=True)
except ImportError as exc:
    raise ImportError("T5Tokenizer needs sentencepiece. In Colab run: !pip -q install sentencepiece") from exc

dtype = torch.float16 if device.type == "cuda" else torch.float32


def load_prostt5(**kwargs):
    try:
        return AutoModelForSeq2SeqLM.from_pretrained(PROSTT5_NAME, dtype=dtype, **kwargs)
    except TypeError:
        return AutoModelForSeq2SeqLM.from_pretrained(PROSTT5_NAME, torch_dtype=dtype, **kwargs)


if device.type == "cuda":
    model = load_prostt5(low_cpu_mem_usage=True, device_map={"": str(device)})
else:
    model = load_prostt5(low_cpu_mem_usage=True).to(device)
model.eval()
model.config.use_cache = True
model.generation_config.use_cache = True
def clean_3di_prediction(decoded: str, aa_length: int) -> str:
    compact = "".join(decoded.split()).replace("<AA2fold>", "").lower()
    states = "".join(char for char in compact if char in THREEDI_ALPHABET)
    return states[:aa_length]


@torch.inference_mode()
def fold_aa_to_3di_prostt5(aa_seq: str) -> str:
    """Generate one 3Di state per AA with the full ProstT5 encoder-decoder."""
    inputs = tokenizer([format_aa(aa_seq)], add_special_tokens=True, return_tensors="pt").to(device)
    prediction = ""
    for extra_tokens in (8, 32, 96):
        output = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            min_new_tokens=len(aa_seq),
            max_new_tokens=len(aa_seq) + extra_tokens,
            do_sample=False,
            num_beams=1,
            use_cache=True,
        )
        decoded = tokenizer.decode(output[0], skip_special_tokens=True)
        prediction = clean_3di_prediction(decoded, len(aa_seq))
        if len(prediction) == len(aa_seq):
            break
    if len(prediction) != len(aa_seq):
        raise RuntimeError(f"ProstT5 length mismatch after retries: AA={len(aa_seq)}, 3Di={len(prediction)}")
    unexpected = sorted(set(prediction) - set(THREEDI_ALPHABET))
    if unexpected:
        raise RuntimeError(f"Unexpected ProstT5 3Di symbols: {unexpected}")
    return prediction

print(f"Loaded full ProstT5 encoder-decoder: {PROSTT5_NAME}")
print("Translation method: autoregressive ProstT5 AA → 3Di (no CNN)")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:134: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.40k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/238k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/283 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/5.64G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/5.64G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loaded full ProstT5 encoder-decoder: Rostlab/ProstT5_fp16
Translation method: autoregressive ProstT5 AA → 3Di (no CNN)


## 5. Translate homologs and project 3Di onto query coordinates

Raw homolog 3Di is indexed by homolog residues. Projection walks the existing AA alignment with separate query and homolog residue counters. A 3Di state is copied only where both query and homolog contain a residue.

In [ ]:
def project_homolog_3di_to_query(query_aln: str, homolog_aln: str,
                                  homolog_3di: str, query_length: int) -> str:
    projected = []
    homolog_position = 0
    for query_char, homolog_char in zip(query_aln, homolog_aln):
        homolog_state = None
        if homolog_char != "-":
            if homolog_position < len(homolog_3di):
                homolog_state = homolog_3di[homolog_position]
            homolog_position += 1
        if query_char != "-":
            if isinstance(homolog_state, str) and homolog_state in THREEDI_ALPHABET:
                projected.append(homolog_state)
            else:
                projected.append("-")
    if homolog_position != len(homolog_3di):
        raise ValueError(f"Alignment consumed {homolog_position} residues, expected {len(homolog_3di)}")
    if len(projected) != query_length:
        raise ValueError(f"Projected length {len(projected)} != query length {query_length}")
    return "".join(projected)


def fasta_text(rows, sequence_key: str) -> str:
    return "".join(
        f">{row['name']} api_rank={row['api_rank']} selected_rank={row['selected_rank']}\n"
        f"{row[sequence_key]}\n" for row in rows
    )


def upsert_factory_summary(summary_row: dict) -> None:
    """Checkpoint one query's status immediately for multi-machine/resume runs."""
    summary_path = OUTPUT_BASE / "factory_run_summary.csv"
    new_df = pd.DataFrame([summary_row])
    if summary_path.exists():
        old_df = pd.read_csv(summary_path)
        combined = pd.concat([old_df, new_df], ignore_index=True)
        combined = combined.drop_duplicates(subset=["query_id"], keep="last")
    else:
        combined = new_df
    combined.to_csv(summary_path, index=False)


def upsert_homolog_manifest(query_id: str, query_aa: str, homolog_rows: list[dict], a3m_path: Path) -> None:
    """Record the ranked API homologs selected for this query in the shared folding_MSA CSV."""
    manifest_rows = [
        {
            "query_id": query_id,
            "query_length": len(query_aa),
            "api_rank": row["api_rank"],
            "selected_rank": row["selected_rank"],
            "homolog_id": row["name"],
            "api_header": row["name"],
            "homolog_aa_length": len(row["aa"]),
            "query_coverage": row["query_coverage"],
            "query_identity": row["query_identity"],
            "selected": True,
            "filter_reason": "selected",
            "a3m_path": str(a3m_path),
        }
        for row in homolog_rows
    ]
    new_df = pd.DataFrame(manifest_rows)
    if HOMOLOG_MANIFEST_PATH.exists():
        try:
            old_df = pd.read_csv(HOMOLOG_MANIFEST_PATH)
        except pd.errors.EmptyDataError:
            old_df = pd.DataFrame()
        combined = pd.concat([old_df, new_df], ignore_index=True)
        if not combined.empty:
            combined = combined.drop_duplicates(subset=["query_id", "api_rank"], keep="last")
    else:
        combined = new_df
    combined.to_csv(HOMOLOG_MANIFEST_PATH, index=False)


def load_converted_rows(output_dir: Path) -> list[dict]:
    csv_path = output_dir / "homologs_aa_3di.csv"
    if not csv_path.exists():
        return []
    try:
        rows = pd.read_csv(csv_path).to_dict("records")
    except pd.errors.EmptyDataError:
        return []
    for row in rows:
        if "selected_rank" in row and pd.notna(row["selected_rank"]):
            row["selected_rank"] = int(row["selected_rank"])
        if "api_rank" in row and pd.notna(row["api_rank"]):
            row["api_rank"] = int(row["api_rank"])
    return rows


def load_failed_rows(output_dir: Path) -> list[dict]:
    csv_path = output_dir / "failed_homologs.csv"
    if not csv_path.exists():
        return []
    try:
        rows = pd.read_csv(csv_path).to_dict("records")
    except pd.errors.EmptyDataError:
        return []
    for row in rows:
        if "selected_rank" in row and pd.notna(row["selected_rank"]):
            row["selected_rank"] = int(row["selected_rank"])
        if "api_rank" in row and pd.notna(row["api_rank"]):
            row["api_rank"] = int(row["api_rank"])
    return rows


def save_failed_rows(output_dir: Path, failed_rows: list[dict]) -> None:
    pd.DataFrame(failed_rows).to_csv(output_dir / "failed_homologs.csv", index=False)


def is_retryable_failed_homolog(row: dict) -> bool:
    message = str(row.get("error_message", ""))
    return "ProstT5 length mismatch" in message


def record_failed_homolog(output_dir: Path, query_id: str, row: dict, exc: Exception,
                          failed_rows: list[dict]) -> None:
    failed_rows[:] = [
        failed for failed in failed_rows
        if failed.get("query_id") != query_id or failed.get("selected_rank") != row["selected_rank"]
    ]
    failed_rows.append({
        "query_id": query_id,
        "api_rank": row["api_rank"],
        "selected_rank": row["selected_rank"],
        "homolog_id": row["name"],
        "homolog_aa_length": len(row["aa"]),
        "query_coverage": row["query_coverage"],
        "query_identity": row["query_identity"],
        "error_type": type(exc).__name__,
        "error_message": str(exc),
    })
    save_failed_rows(output_dir, failed_rows)


def save_query_checkpoint(query_id: str, query_aa: str, query_3di: str | None,
                          converted_rows: list[dict], usable_homologs: int,
                          a3m_path: Path, output_dir: Path, status: str) -> dict:
    """Write all currently converted homolog rows immediately for crash-safe multi-machine runs."""
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / "query_aa.fasta").write_text(f">{query_id}\n{query_aa}\n", encoding="utf-8")
    if query_3di is not None:
        (output_dir / "query_3di.fasta").write_text(f">{query_id}\n{query_3di}\n", encoding="utf-8")
    (output_dir / "homologs_aa.fasta").write_text(fasta_text(converted_rows, "aa"), encoding="utf-8")
    (output_dir / "homologs_raw_3di.fasta").write_text(fasta_text(converted_rows, "raw_3di"), encoding="utf-8")
    (output_dir / "homologs_projected_to_query_3di.fasta").write_text(
        fasta_text(converted_rows, "projected_3di"), encoding="utf-8",
    )
    pd.DataFrame(converted_rows).to_csv(output_dir / "homologs_aa_3di.csv", index=False)
    checkpoint_csv = output_dir / "homologs_aa_3di.csv"
    projected_fasta = output_dir / "homologs_projected_to_query_3di.fasta"
    if not checkpoint_csv.exists() or not projected_fasta.exists():
        raise RuntimeError(f"Checkpoint write verification failed for {output_dir}")
    summary_row = {
        "query_id": query_id,
        "status": status,
        "usable_homologs": usable_homologs,
        "converted_homologs": len(converted_rows),
        "a3m_path": str(a3m_path),
        "output_dir": str(output_dir),
    }
    pd.DataFrame([summary_row]).to_csv(output_dir / "factory_status.csv", index=False)
    upsert_factory_summary(summary_row)
    print(f"Checkpoint verified: {checkpoint_csv} ({len(converted_rows)} homolog rows)")
    return summary_row


run_summary = []
for query_index, item in enumerate(pending_queries, start=1):
    query_id = item["query_id"]
    query_aa = item["query_aa"]
    a3m_path = MSA_DIR / f"{query_id}.a3m"
    output_dir = OUTPUT_BASE / query_id
    output_dir.mkdir(parents=True, exist_ok=True)
    refresh_cache = a3m_path.exists() and item.get("cached_homologs", 0) < MAX_HOMOLOGS

    print(f"[{query_index}/{len(pending_queries)}] {query_id}")
    a3m_text = fetch_msa_colabfold(
        query_aa, a3m_path, mode=MSA_MODE, force_refresh=refresh_cache,
    )
    query_alignment, homolog_rows = select_ranked_homologs(
        a3m_text, query_aa, max_homologs=None,
    )
    upsert_homolog_manifest(query_id, query_aa, homolog_rows, a3m_path)

    converted_rows = load_converted_rows(output_dir)
    failed_rows = load_failed_rows(output_dir)
    converted_ranks = {row["selected_rank"] for row in converted_rows if "selected_rank" in row}
    failed_ranks = {
        row["selected_rank"] for row in failed_rows
        if "selected_rank" in row and not is_retryable_failed_homolog(row)
    }
    rows_to_convert = [
        row for row in homolog_rows
        if row["selected_rank"] not in converted_ranks and row["selected_rank"] not in failed_ranks
    ]
    if converted_rows:
        print(f"Resume {query_id}: {len(converted_rows)} homolog 3Di rows already saved")
    if failed_rows:
        print(f"Resume {query_id}: {len(failed_rows)} failed homologs already skipped")
    save_query_checkpoint(
        query_id, query_aa, None, converted_rows, len(homolog_rows),
        a3m_path, output_dir, "partial" if len(converted_rows) < MAX_HOMOLOGS else "processed",
    )

    for row in tqdm(rows_to_convert, desc=f"{query_id} AA → 3Di", leave=False):
        if len(converted_rows) >= MAX_HOMOLOGS:
            break
        try:
            raw_3di = fold_aa_to_3di_prostt5(row["aa"])
            projected_3di = project_homolog_3di_to_query(
                query_alignment, row["aa_alignment"], raw_3di, len(query_aa),
            )
        except Exception as exc:
            print(
                f"SKIP failed homolog {query_id} api_rank={row['api_rank']} "
                f"selected_rank={row['selected_rank']} length={len(row['aa'])}: {exc}"
            )
            record_failed_homolog(output_dir, query_id, row, exc, failed_rows)
            continue
        failed_rows[:] = [
            failed for failed in failed_rows
            if failed.get("query_id") != query_id or failed.get("selected_rank") != row["selected_rank"]
        ]
        save_failed_rows(output_dir, failed_rows)
        converted_rows.append({
            **row,
            "query_id": query_id,
            "converted_rank": len(converted_rows) + 1,
            "raw_3di": raw_3di,
            "projected_3di": projected_3di,
            "projected_coverage": sum(c != "-" for c in projected_3di) / len(projected_3di),
            "translator": "ProstT5 encoder-decoder",
            "translator_model": PROSTT5_NAME,
        })
        save_query_checkpoint(
            query_id, query_aa, None, converted_rows, len(homolog_rows),
            a3m_path, output_dir, "partial",
        )

    if len(converted_rows) < MAX_HOMOLOGS:
        print(
            f"WARNING {query_id}: only {len(converted_rows)}/{MAX_HOMOLOGS} homologs converted "
            f"after {len(failed_rows)} failures and {len(homolog_rows)} usable candidates."
        )

    query_3di_path = output_dir / "query_3di.fasta"
    if query_3di_path.exists():
        query_3di = "".join(
            line.strip() for line in query_3di_path.read_text(encoding="utf-8").splitlines()
            if line.strip() and not line.startswith(">")
        )
    else:
        query_3di = fold_aa_to_3di_prostt5(query_aa)
    final_status = "processed" if len(converted_rows) >= MAX_HOMOLOGS else "insufficient_homologs"
    summary_row = save_query_checkpoint(
        query_id, query_aa, query_3di, converted_rows, len(homolog_rows),
        a3m_path, output_dir, final_status,
    )
    run_summary.append(summary_row)
    print(f"Saved {query_id} outputs immediately to {output_dir}")

run_summary_df = pd.DataFrame(run_summary)
run_summary_df


[1/20] P61583
Resume P61583: 29 homolog 3Di rows already saved
Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/P61583/homologs_aa_3di.csv (29 homolog rows)


P61583 AA → 3Di: 0it [00:00, ?it/s]

WARNING P61583: only 29/64 homologs converted after 0 failures and 29 usable candidates.
Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/P61583/homologs_aa_3di.csv (29 homolog rows)
Saved P61583 outputs immediately to /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/P61583
[2/20] P42336
Using cached A3M: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/P42336.a3m
Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/P42336/homologs_aa_3di.csv (0 homolog rows)


P42336 AA → 3Di:   0%|          | 0/3260 [00:00<?, ?it/s]

Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/P42336/homologs_aa_3di.csv (1 homolog rows)
Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/P42336/homologs_aa_3di.csv (2 homolog rows)
Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/P42336/homologs_aa_3di.csv (3 homolog rows)
Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/P42336/homologs_aa_3di.csv (4 homolog rows)
Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/P42336/homologs_aa_3di.csv (5 homolog rows)
Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/P42336/homologs_aa_3di.csv (6 homolog rows)
Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding

...

Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/P02458/homologs_aa_3di.csv (1 homolog rows)
Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/P02458/homologs_aa_3di.csv (2 homolog rows)
Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/P02458/homologs_aa_3di.csv (3 homolog rows)
Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/P02458/homologs_aa_3di.csv (4 homolog rows)
Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/P02458/homologs_aa_3di.csv (5 homolog rows)
Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/P02458/homologs_aa_3di.csv (6 homolog rows)
Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding

Q61410 AA → 3Di:   0%|          | 0/3379 [00:00<?, ?it/s]

Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/Q61410/homologs_aa_3di.csv (1 homolog rows)
Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/Q61410/homologs_aa_3di.csv (2 homolog rows)
Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/Q61410/homologs_aa_3di.csv (3 homolog rows)
Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/Q61410/homologs_aa_3di.csv (4 homolog rows)
Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/Q61410/homologs_aa_3di.csv (5 homolog rows)
Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/Q61410/homologs_aa_3di.csv (6 homolog rows)
Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding

P13368 AA → 3Di:   0%|          | 0/161 [00:00<?, ?it/s]

Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/P13368/homologs_aa_3di.csv (1 homolog rows)
Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/P13368/homologs_aa_3di.csv (2 homolog rows)
Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/P13368/homologs_aa_3di.csv (3 homolog rows)
Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/P13368/homologs_aa_3di.csv (4 homolog rows)
Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/P13368/homologs_aa_3di.csv (5 homolog rows)
Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding-ProstT5/prostT5/prostT5_benchmarks/folding_MSA/P13368/homologs_aa_3di.csv (6 homolog rows)
Checkpoint verified: /content/drive/MyDrive/Speculative-Decoding

## 6. Save the factory outputs

The homolog files contain homologs only. The query is saved separately, so it cannot accidentally be counted as family evidence.

In [ ]:
skipped_summary = pd.DataFrame([
    {
        "query_id": row["query_id"],
        "status": "skipped_complete_3di",
        "usable_homologs": row["usable_homologs"],
        "converted_homologs": row["converted_homologs"],
        "a3m_path": str(MSA_DIR / f"{row['query_id']}.a3m"),
        "output_dir": str(OUTPUT_BASE / row["query_id"]),
    }
    for row in skipped_queries
])
summary_path = OUTPUT_BASE / "factory_run_summary.csv"
new_summary_df = pd.concat([skipped_summary, run_summary_df], ignore_index=True)
if summary_path.exists():
    old_summary_df = pd.read_csv(summary_path)
    all_summary_df = pd.concat([old_summary_df, new_summary_df], ignore_index=True)
    all_summary_df = all_summary_df.drop_duplicates(subset=["query_id"], keep="last")
else:
    all_summary_df = new_summary_df
all_summary_df.to_csv(summary_path, index=False)
print(f"Saved run summary: {summary_path}")
print(f"Skipped: {len(skipped_queries)}; processed: {len(run_summary)}")
all_summary_df


## 7. Colab download helper

When running on Colab, checkpoints are written to Google Drive. This cell also creates a zip archive that can be downloaded and copied into the local Mac `folding_MSA` folder.

In [ ]:
if running_in_colab():
    archive_base = OUTPUT_BASE.parent / "folding_MSA_colab_export"
    archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_BASE))
    print(f"Created Colab export zip: {archive_path}")
    print(f"Download this zip, unzip it, and copy/merge contents into: {LOCAL_MSA_DIR}")
    try:
        from google.colab import files
        files.download(str(archive_path))
    except Exception as exc:
        print(f"Automatic browser download unavailable: {exc}")
else:
    print(f"Local run: outputs already saved at {OUTPUT_BASE}")


## Alignment decision

Use the MMseqs2/ColabFold A3M alignment as the alignment source. Do not run an unrelated pairwise aligner after converting each homolog to 3Di: that can change the residue correspondence. For HMM/profile construction, use `projected_3di`, because every row then has exactly the query length and `-` marks missing homolog coverage. Keep `raw_3di` as the unmodified ProstT5 output for auditing.

The query must not be counted as a homolog. This notebook skips the first A3M query row and also removes any later row whose ungapped AA sequence exactly equals the query.